# Asignación de Polos — Diseño de Controladores

Este notebook muestra cómo usar la función `asigne_polos` para diseñar un controlador
por asignación de polos mediante la **matriz de Sylvester**, y visualizar los resultados.

**Esquema de control (realimentación unitaria):**

```
  r ──►(+)──► C(s) ──► G(s) ──► y
        ▲ −                │
        └──────────────────┘
```


In [ ]:
!pip install control
import numpy as np
import matplotlib.pyplot as plt
import control as ctrl

# %!wget -O metodo_algebraico.py https://raw.githubusercontent.com/nebisman/control-material/refs/heads/main/notebooks/metodo_algebraico.py
from metodo_algebraico import asigne_polos, calcular_itae

%matplotlib inline
plt.rcParams.update({
    'figure.figsize': (10, 4),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2,
    'font.size': 11,
})


## 1. Definimos la planta



$$G(s) = \frac{395}{s(0.141s+1)} $$

Polos deseados:

$$\{-\zeta \omega_n + \omega_n \sqrt{1-\zeta^2}  , -\zeta \omega_n + \omega_n \sqrt{1-\zeta^2} , -3\omega_n\}$$

Deben ser 3 porque el controlador de orden minimo es 1.

In [ ]:
G = ctrl.tf([395], [0.141, 1, 0])
print("Planta G(s):")
print(G)

## 2. Elegir los polos deseados

Para una planta de orden $n = 2$, necesitamos al menos $2n - 1 = 3$ polos para un controlador de orden minimo. Por ejemplo con $\omega_n=1$:

$$\{-\zeta \omega_n + \omega_n \sqrt{1-\zeta^2}  , -\zeta \omega_n + \omega_n \sqrt{1-\zeta^2} , -3\omega_n\}$$

In [ ]:
wn=10
z=0.69
polos_deseados = [-z*wn+1j*wn*np.sqrt(1-z**2), -z*wn-1j*wn*np.sqrt(1-z**2), -3*wn]

print(f"Polos deseados: {polos_deseados}")


## 3. Diseñar el controlador

In [ ]:
C, T, Gur = asigne_polos(G, polos_deseados)

print("Controlador C(s):")
print(C)
print()
print("Lazo cerrado T(s):")
print(T)
print(f"\nPolos obtenidos: {np.round(ctrl.poles(T), 4)}")


## 4. Respuesta al escalón

Obtenemos la respuesta en lazo cerrado del sistema.


In [ ]:
# respuesta al escalón
ref = 100
t_cl, y_cl = ctrl.step_response(ref*T)



fig, axes = plt.subplots(1, 2, figsize=(12, 4))
t_u, y_u = ctrl.step_response(ref*Gur, t_cl)

ax = axes[0]
ax.plot(t_cl, y_cl, "-",  color="#0066cc", lw=2.5, label="Lazo cerrado T(s)")
ax.axhline(1, color="gray", lw=0.6, ls=":")
ax.set_xlabel("Tiempo [s]")
ax.set_ylabel("y(t)")
ax.set_title("Respuesta al escalón — Planta de 3er orden")
ax.legend()

ax = axes[1]
ax.plot(t_u, y_u, "-", color="#cc3300", lw=2, label="u(t)")
ax.set_xlabel("Tiempo [s]")
ax.set_ylabel("u(t)")
ax.set_title("Señal de  control")
ax.legend()
plt.tight_layout()
plt.show()


# Diseño con polos itae
- Ahora vamos a diseñar un sistema con polos ITAE para simplificar el diseño



In [ ]:
w = 10
ref = 10

T_itae = calcular_itae(3,w)
polos = T_itae.poles()
print(f"Polos: {polos}")

# diseñamos con itae
C, T, Gur = asigne_polos(G, polos)
print("Controlador C(s):")
print(C)
print()
print("Lazo cerrado T(s):")
print(T)



# respuesta al escalón
t_cl, y_cl = ctrl.step_response(ref*T,5)



fig, axes = plt.subplots(1, 2, figsize=(12, 4))
t_u, y_u = ctrl.step_response(ref*Gur, t_cl)

ax = axes[0]
ax.plot(t_cl, y_cl, "-",  color="#0066cc", lw=2.5, label="Lazo cerrado T(s)")
ax.axhline(1, color="gray", lw=0.6, ls=":")
ax.set_xlabel("Tiempo [s]")
ax.set_ylabel("y(t)")
ax.set_title("Respuesta al escalón — Planta de 2do orden")
ax.legend()

ax = axes[1]
ax.plot(t_u, y_u, "-", color="#cc3300", lw=2, label="u(t)")
ax.set_xlabel("Tiempo [s]")
ax.set_ylabel("u(t)")
ax.set_title("Señal de  control")
ax.legend()
plt.tight_layout()
plt.show()

